# SOFR, FX & Cross-Currency Curves · Episode 2 — How SOFR compounds

Companion notebook for the video. We settle one real-dated SOFR overnight index swap three ways: by hand, with QuantLib, and with the New York Fed's SOFR Index. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Data.** `data/nyfed_sofr_index.csv`: daily SOFR and the SOFR Index from the New York Fed's markets data API (https://markets.newyorkfed.org).
>
> The SOFR data are subject to the Terms of Use posted at newyorkfed.org. The New York Fed is not responsible for publication of the SOFR data by Rates with Bip, does not sanction or endorse any particular republication, and has no liability for your use. The SOFR data are calculated using data provided under a license granted to the New York Fed by DTCC Solutions LLC ("Solutions"), an affiliate of The Depository Trust & Clearing Corporation; Solutions, its affiliates, and third parties from which they obtained data have no liability for the content of this material. Rates with Bip is not affiliated with the New York Fed. The New York Fed does not sanction, endorse, or recommend any products or services offered by Rates with Bip.
>
> **Illustrative trade.** The fixed rate is made up for teaching; the SOFR fixings are real. Educational material only, not investment advice.

1. Setup · 2. The trade · 3. Daily fixings · 4. Compounding by hand · 5. QuantLib · 6. Check against the SOFR Index · 7. Settlement · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `data/nyfed_sofr_index.csv` (run me first) { display-mode: "form" }
# Source: Federal Reserve Bank of New York (SOFR, EFFR). The SOFR and EFFR data are subject to the Terms of
# Use posted at newyorkfed.org. The New York Fed is not responsible for publication of the SOFR and EFFR data
# by Rates with Bip, does not sanction or endorse any particular republication, and has no liability for
# your use. SOFR data are calculated using data provided under a license granted to the New York Fed by
# DTCC Solutions LLC; Solutions, its affiliates, and third parties from which they obtained data have no
# liability for the content of this material. Rates with Bip is not affiliated with the New York Fed.
from pathlib import Path
Path('data/nyfed_sofr_index.csv').parent.mkdir(parents=True, exist_ok=True)
Path('data/nyfed_sofr_index.csv').write_text("""date,sofr,sofr_index
2026-05-01,3.64,1.24161826
2026-05-04,3.63,1.24199489
2026-05-05,3.62,1.24212012
2026-05-06,3.61,1.24224503
2026-05-07,3.6,1.2423696
2026-05-08,3.6,1.24249383
2026-05-11,3.6,1.24286658
2026-05-12,3.6,1.24299087
2026-05-13,3.59,1.24311517
2026-05-14,3.56,1.24323913
2026-05-15,3.55,1.24336207
2026-05-18,3.53,1.2437299
2026-05-19,3.51,1.24385186
2026-05-20,3.5,1.24397313
2026-05-21,3.51,1.24409407
2026-05-22,3.55,1.24421537
2026-05-26,3.63,1.24470615
2026-05-27,3.63,1.24483166
2026-05-28,3.62,1.24495718
2026-05-29,3.63,1.24508236
2026-06-01,3.65,1.245459
2026-06-02,3.63,1.24558528
2026-06-03,3.61,1.24571087
2026-06-04,3.62,1.24583579
2026-06-05,3.63,1.24596107
2026-06-08,3.63,1.24633797
2026-06-09,3.6,1.24646364
2026-06-10,3.59,1.24658829
2026-06-11,3.6,1.2467126
2026-06-12,3.65,1.24683727
2026-06-15,3.69,1.24721652
2026-06-16,3.63,1.24734436
2026-06-17,3.63,1.24747013
2026-06-18,3.62,1.24759592
2026-06-22,3.61,1.24809773
2026-06-23,3.62,1.24822289
2026-06-24,3.62,1.2483484
2026-06-25,3.64,1.24847393
2026-06-26,3.62,1.24860016
2026-06-29,3.62,1.24897683
2026-06-30,3.68,1.24910242
2026-07-01,3.66,1.2492301
2026-07-02,3.64,1.24935711
2026-07-06,3.63,1.2498624
2026-07-07,3.62,1.24998843
2026-07-08,3.58,1.25011412
2026-07-09,3.53,1.25023844
2026-07-10,3.55,1.25036103
2026-07-13,3.6,1.25073093
2026-07-14,3.63,1.25085601
2026-07-15,3.64,1.25098213
2026-07-16,3.62,1.25110862
2026-07-17,3.59,1.25123443
2026-07-20,3.57,1.25160876
2026-07-21,3.61,1.25173287
2026-07-22,3.62,1.25185839
2026-07-23,3.64,1.25198428
2026-07-24,3.64,1.25211087
2026-07-27,3.64,1.25249067
2026-07-28,3.65,1.25261731
2026-07-29,3.65,1.25274431
2026-07-30,3.65,1.25287133
2026-07-31,3.66,1.25299836
2026-08-03,3.65,1.25338052
2026-08-04,3.66,1.2535076
2026-08-05,3.64,1.25363504
2026-08-06,3.65,1.2537618
2026-08-07,3.62,1.25388891
2026-08-10,3.63,1.25426717
2026-08-11,3.64,1.25439364
2026-08-12,3.62,1.25452047
2026-08-13,3.62,1.25464662
2026-08-14,3.62,1.25477279
2026-08-17,3.66,1.25515131
2026-08-18,3.65,1.25527892
2026-08-19,3.62,1.25540619
2026-08-20,3.63,1.25553243
2026-08-21,3.65,1.25565902
2026-08-24,3.65,1.25604095
2026-08-25,3.66,1.2561683
2026-08-26,3.64,1.25629601
2026-08-27,3.64,1.25642304
2026-08-28,3.65,1.25655008
2026-08-31,3.68,1.25693228
2026-09-01,3.66,1.25706076
2026-09-02,3.65,1.25718857
2026-09-03,3.66,1.25731603
2026-09-04,3.65,1.25744386
2026-09-08,3.64,1.25795382
2026-09-09,3.64,1.25808101
2026-09-10,3.62,1.25820822
2026-09-11,3.62,1.25833474
2026-09-14,3.62,1.25871434
2026-09-15,3.64,1.25884091
2026-09-16,3.62,1.25896819
2026-09-17,3.85,1.25909479
2026-09-18,3.85,1.25922944
2026-09-21,3.85,1.25963344
2026-09-22,3.87,1.25976815
2026-09-23,3.87,1.25990358
2026-09-24,3.88,1.26003902
2026-09-25,,1.26017482
""")
print('wrote data/nyfed_sofr_index.csv')

In [ ]:
# Parameters (papermill overrides these)
trade_date = "2026-06-15"
tenor = "3M"
notional = 100_000_000
fixed_rate_pct = 3.60          # illustrative
data_csv = "data/nyfed_sofr_index.csv"
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

dc = ql.Actual360()
iso = lambda d: d.ISO()
pydate = lambda d: dt.date(d.year(), d.month(), d.dayOfMonth())

pay_cal = ql.UnitedStates(ql.UnitedStates.Settlement)   # stand-in for New York business days
sofr = ql.Sofr()                                        # fixings on the US government securities calendar
fix_cal = sofr.fixingCalendar()

data = pd.read_csv(data_csv, parse_dates=["date"])
data["date"] = data.date.dt.date
rate_on = data.dropna(subset=["sofr"]).set_index("date").sofr
index_on = data.set_index("date").sofr_index

out = {"meta": {
    "series": "SOFR, FX & Cross-Currency Curves", "episode": 2, "valuation_date": trade_date,
    "quantlib_version": ql.__version__,
    "quotes_label": "real SOFR fixings and SOFR Index (New York Fed); illustrative fixed rate",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| SOFR data to", rate_on.index.max(), "| fixing calendar:", fix_cal.name())

## 2. The trade

Receive fixed on a 3-month SOFR OIS. Conventions (CFTC *Swaps Made Available to Trade*, USD SOFR OIS): spot start two business days after the trade (T+2), New York business days, ACT/360 on both legs, one payment (annual legs, so a 3-month swap pays once), paid two business days after the end date; SOFR fixings follow the US government securities calendar. The end date rolls Modified Following.

In [ ]:
trade = ql.DateParser.parseISO(trade_date)
start = pay_cal.advance(trade, 2, ql.Days)
end = pay_cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
pay = pay_cal.advance(end, 2, ql.Days)
days = end - start
K = fixed_rate_pct / 100
out["trade"] = {"trade_date": iso(trade), "start": iso(start), "end": iso(end), "pay": iso(pay), "days": days,
                "tenor": tenor, "notional": notional, "fixed_rate_pct": fixed_rate_pct, "side": "receive fixed"}
pd.Series(out["trade"])

## 3. Daily fixings

Each US government securities business day $d_i$ from the start date up to, but not including, the end date has a fixing $r_i$: the SOFR the New York Fed publishes for that day (the next business day, about 8:00am ET). It accrues for $n_i$ calendar days, until the next such business day, so a Friday rate usually counts three times and the day before a holiday counts more.

The fixing calendar is not the Fed's: on Friday 3 July 2026 the Fed was open but the government securities market was closed, so there is no SOFR for that day. We check that QuantLib's SOFR calendar gives exactly the days the New York Fed published.

In [ ]:
fix_days = []
d = start
while d < end:
    if fix_cal.isBusinessDay(d):
        fix_days.append(d)
    d += 1

rows = []
for i, d in enumerate(fix_days):
    nxt = fix_days[i + 1] if i + 1 < len(fix_days) else end
    rows.append({"date": iso(d), "weekday": pydate(d).strftime("%a"), "rate_pct": float(rate_on[pydate(d)]),
                 "days": nxt - d})
fx = pd.DataFrame(rows)
fx["factor"] = 1 + fx.rate_pct / 100 * fx.days / 360
fx["cumulative"] = fx.factor.cumprod()

published = sorted(x for x in rate_on.index if pydate(start) <= x < pydate(end))
assert [dt.date.fromisoformat(r) for r in fx.date] == published, "fixing days differ from the days the New York Fed published"
fed_open_no_sofr = [iso(d) for d in [ql.Date(3, 7, 2026)] if ql.UnitedStates(ql.UnitedStates.FederalReserve).isBusinessDay(d)
                    and not fix_cal.isBusinessDay(d)]
out["fixings"] = {
    "n": len(fx), "days_total": int(fx.days.sum()), "rows": fx.to_dict("records"),
    "matches_nyfed_publication_days": True,
    "long_weights": fx[fx.days > 3].to_dict("records"),
    "first_rate_pct": float(fx.rate_pct.iloc[0]), "last_rate_pct": float(fx.rate_pct.iloc[-1]),
    "min_rate_pct": float(fx.rate_pct.min()), "max_rate_pct": float(fx.rate_pct.max()),
    "max_date": fx.date[fx.rate_pct.idxmax()],
    "last_date": fx.date.iloc[-1], "last_published": iso(fix_cal.advance(fix_days[-1], 1, ql.Days)),
    "fed_open_no_sofr": fed_open_no_sofr,
}
fx.head(8)

A few rows for the video: the first day, the days that cover long weekends, the highest fixing and the last day.

In [ ]:
pick = sorted(set([0, len(fx) - 1, int(fx.rate_pct.idxmax())] + list(fx.index[fx.days > 3])))
sample = fx.loc[pick].copy()
sample["label"] = sample.date.map(lambda s: (lambda d: f"{d:%a} {d.day} {d:%b}")(dt.date.fromisoformat(s)))
out["fixings"]["sample"] = sample.to_dict("records")
sample

## 4. Compounding by hand

$$\text{Float} = N\left[\prod_{i}\left(1 + r_i\,\frac{n_i}{360}\right) - 1\right], \qquad \text{Fixed} = N \cdot K \cdot \frac{D}{360}$$

where $D$ is the number of days from start to end.

In [ ]:
growth = float(fx.factor.prod())
float_hand = notional * (growth - 1)
fixed_amt = notional * K * days / 360
compounded_pct = (growth - 1) * 360 / days * 100
average_pct = float((fx.rate_pct * fx.days).sum() / days)
hand = {
    "growth": growth, "float_amount": float_hand, "float_amount_cents": round(float_hand, 2),
    "fixed_amount": fixed_amt, "fixed_amount_cents": round(fixed_amt, 2),
    "compounded_rate_pct": compounded_pct, "average_rate_pct": average_pct,
    "compounding_bp": (compounded_pct - average_pct) * 100,
    "float_simple_amount": notional * average_pct / 100 * days / 360,
}
hand["compounding_amount"] = hand["float_amount"] - hand["float_simple_amount"]
out["hand"] = hand
pd.Series(hand)

## 5. QuantLib

Load the New York Fed fixings into QuantLib's SOFR index, build the swap with an explicit schedule and a two-day payment lag, and set the evaluation date to the end date, so every fixing is in the past and nothing is forecast.

In [ ]:
for r in data.dropna(subset=["sofr"]).itertuples():
    sofr.addFixing(ql.Date(r.date.day, r.date.month, r.date.year), r.sofr / 100)

ql.Settings.instance().evaluationDate = end
schedule = ql.Schedule(start, end, ql.Period(ql.Once), pay_cal, ql.ModifiedFollowing, ql.ModifiedFollowing,
                       ql.DateGeneration.Backward, False)
swap = ql.OvernightIndexedSwap(ql.Swap.Receiver, notional, schedule, K, dc, sofr, 0.0,
                               2, ql.Following, pay_cal)  # payment lag: two business days
fixed_cf, float_cf = swap.fixedLeg()[0], swap.overnightLeg()[0]
coupon = ql.as_overnight_indexed_coupon(float_cf)
qlres = {
    "float_amount": float_cf.amount(), "fixed_amount": fixed_cf.amount(),
    "compounded_rate_pct": coupon.rate() * 100, "payment_date": iso(float_cf.date()),
    "n_fixings": len(coupon.fixingDates()),
}
qlres["abs_diff_float"] = abs(qlres["float_amount"] - hand["float_amount"])
out["quantlib"] = qlres
pd.Series(qlres)

## 6. Check against the New York Fed's SOFR Index

The SOFR Index measures the cumulative effect of compounding SOFR on each business day, starting from 1.00000000 on 2 April 2018; it is published on its value date and includes the SOFR published that day (New York Fed, *Additional Information about Reference Rates*). So the index on the end date over the index on the start date is the growth of the floating leg. The index has eight decimal places, so expect agreement to within about a dollar on USD 100 million.

In [ ]:
idx_growth = float(index_on[pydate(end)] / index_on[pydate(start)])
idx = {"index_start": float(index_on[pydate(start)]), "index_end": float(index_on[pydate(end)]),
       "growth": idx_growth, "float_amount": notional * (idx_growth - 1)}
idx["abs_diff_float"] = abs(idx["float_amount"] - hand["float_amount"])
idx["compounded_rate_pct"] = (idx_growth - 1) * 360 / days * 100
out["index"] = idx
pd.Series(idx)

## 7. Settlement

The receiver of fixed gets the fixed amount and pays the floating amount; the difference is paid two business days after the end date.

In [ ]:
net = round(fixed_amt, 2) - round(float_hand, 2)
out["settlement"] = {"net_to_receiver": net, "payer": "fixed receiver" if net < 0 else "fixed payer",
                     "abs_net": abs(net), "pay_date": iso(pay), "breakeven_fixed_pct": compounded_pct}
pd.Series(out["settlement"])

Chart data: the daily rate and the running compounded rate over the life of the swap.

In [ ]:
x = [(dt.date.fromisoformat(r) - pydate(start)).days for r in fx.date] + [days]
running = []
acc, elapsed = 1.0, 0
for r in fx.itertuples():
    acc *= r.factor
    elapsed += r.days
    running.append((acc - 1) * 360 / elapsed * 100)
ticks = []
m = dt.date(pydate(start).year, pydate(start).month, 1)
while m <= pydate(end):
    if m > pydate(start):
        ticks.append({"x": (m - pydate(start)).days, "label": m.strftime("1 %b")})
    m = dt.date(m.year + (m.month == 12), m.month % 12 + 1, 1)
out["chart"] = {
    "x": x, "rate_pct": fx.rate_pct.tolist() + [fx.rate_pct.iloc[-1]],
    "running_pct": running + [running[-1]],
    "fixed_pct": [fixed_rate_pct] * (len(x)),
    "ticks": [{"x": 0, "label": pydate(start).strftime("%d %b").lstrip("0")}] + ticks
             + [{"x": days, "label": pydate(end).strftime("%d %b").lstrip("0")}],
    "range": [0, days],
}
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(out["chart"]["x"], out["chart"]["rate_pct"], where="post", label="SOFR (Source: New York Fed)")
ax.plot(out["chart"]["x"], out["chart"]["running_pct"], "--", label="compounded so far, annualised")
ax.axhline(fixed_rate_pct, color="k", lw=1, ls=":", label="fixed rate (illustrative)")
ax.set(xlabel="days since start", ylabel="%")
ax.grid(alpha=.3)
ax.legend()
plt.tight_layout()

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")